**Cloning the Jacobian Lens Repo**

In [ ]:
!git clone --depth 1 https://github.com/anthropics/jacobian-lens


**Installing Jacobian Lens**

In [ ]:
!pip install -e /content/jacobian-lens

import sys; sys.path.append("/content/jacobian-lens")

%cd jacobian-lens

**Loading JLens for Qwen 3.5 and testing**

In [ ]:
import torch, transformers, jlens

assert torch.cuda.is_available(), "Need a GPU runtime"

hf = transformers.AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen3.5-4B", dtype="float16", device_map="cuda")
tok = transformers.AutoTokenizer.from_pretrained("Qwen/Qwen3.5-4B")


model = jlens.from_hf(hf, tok)

lens = jlens.JacobianLens.from_pretrained(
    "neuronpedia/jacobian-lens",
    filename="qwen3.5-4b/jlens/Salesforce-wikitext/Qwen3.5-4B_jacobian_lens_n1000.pt",
    revision="qwen-n1000")



**Walkthrough**

In [ ]:
import json, os, inspect
from IPython.display import display
from jlens.vis import build_page, compute_slice, notebook_iframe

path = "/content/Gender_identity.jsonl"
if not os.path.exists(path):
    !wget -qO {path} https://raw.githubusercontent.com/nyu-mll/BBQ/main/data/Gender_identity.jsonl

kw = dict(layer_stride=1, mask_display=True)
for a in ("top_n", "top_k", "k"):
    if a in inspect.signature(compute_slice).parameters:
        kw[a] = 15
        break

WANT = {0, 1, 24, 25}
slices, prompts, questions = {}, {}, {}
for line in open(path):
    r = json.loads(line)
    if r["example_id"] not in WANT:
        continue
    n = str(r["example_id"])
    opts = "\n".join(f'{c}. {r[f"ans{i}"]}' for i, c in enumerate("ABC"))
    prompts[n] = f'{r["context"]} {r["question"]}\n{opts}\nAnswer:'
    questions[n] = r["question"]
    slices[n] = compute_slice(model, lens, prompts[n], **kw)
    print(n, "|", slices[n].seq_len, "tokens |", r["context_condition"])
    if len(slices) == len(WANT):
        break

for n, s in slices.items():
    page, _, _ = build_page(s, prompts[n], title=n, description=questions[n])
    display(notebook_iframe(page))

**Attempt at new viz:** Interactive display allowing user to visualize a list of tokens adjusted by topk, position range, and layer range.

The list contains the number of times a token appears in those ranges. User can click on the token to display the layers and positions in which the token appears.

Choose item based on example id number (0, 1, 24, or 25)





In [ ]:
import json
from IPython.display import HTML, display

s0 = next(iter(slices.values()))
K = int(min(15, s0.top_ids.shape[-1]))
L = [int(x) for x in s0.layers]
D = {n: [[[tok.decode([int(t)]) for t in s.top_ids[p, j][:K]]
          for j in range(len(s.layers))] for p in range(s.seq_len)]
     for n, s in slices.items()}
POS = {n: [s.context_token_strs[p] for p in range(s.seq_len)] for n, s in slices.items()}
print("layers:", len(L), "| topk available:", K)

html = """
<style>
 .row{display:flex;gap:14px;font:12px sans-serif;align-items:center;margin:3px 0}
 .row input[type=range]{flex:1} .tk{cursor:pointer;padding:2px 4px;font:13px monospace}
 .tk:hover{background:#eee} #wrap{display:flex;gap:20px;margin-top:8px}
 #cnt,#det{height:320px;overflow:auto;flex:1;border:1px solid #ccc;padding:6px}
</style>
<div class=row>item <select id=it></select></div>
<div class=row>layers <span id=la></span><input type=range id=l0><input type=range id=l1></div>
<div class=row>positions <span id=pa></span><input type=range id=p0><input type=range id=p1></div>
<div class=row>top-k <span id=ka></span><input type=range id=k min=1 max=__K__ value=__K__></div>
<div id=wrap><div id=cnt></div><div id=det>click a token</div></div>
<script>
const DD=__D__, PP=__POS__, L=__L__, N=Object.keys(DD);
const g=i=>document.getElementById(i), esc=s=>s.replace(/</g,'&lt;');
let D, POS, where={};
g('it').innerHTML=N.map((n,i)=>`<option value=${i}>${n}</option>`).join('');
[['l0',0],['l1',L.length-1]].forEach(([i,v])=>{
  let e=g(i); e.min=0; e.max=L.length-1; e.value=v; e.oninput=draw});
g('k').oninput=draw;
g('p0').oninput=g('p1').oninput=draw;
g('it').onchange=()=>{setup();draw()};
function setup(){
  const n=N[+g('it').value]; D=DD[n]; POS=PP[n];
  g('p0').min=g('p1').min=0; g('p0').max=g('p1').max=POS.length-1;
  g('p0').value=0; g('p1').value=POS.length-1;
}
function draw(){
  const l0=Math.min(+g('l0').value,+g('l1').value), l1=Math.max(+g('l0').value,+g('l1').value);
  const p0=Math.min(+g('p0').value,+g('p1').value), p1=Math.max(+g('p0').value,+g('p1').value);
  const k=+g('k').value;
  g('la').textContent=L[l0]+'-'+L[l1]; g('pa').textContent=p0+'-'+p1; g('ka').textContent=k;
  const c={}; where={};
  for(let p=p0;p<=p1;p++) for(let j=l0;j<=l1;j++)
    D[p][j].slice(0,k).forEach(t=>{c[t]=(c[t]||0)+1;(where[t]=where[t]||[]).push([p,L[j]])});
  const e=Object.entries(c).sort((a,b)=>b[1]-a[1]);
  g('cnt').innerHTML='<b>'+e.length+' distinct</b><br>'+e.map(([t,n],i)=>
    `<div class=tk data-i="${i}">${esc(t)} &nbsp;<b>${n}</b></div>`).join('');
  g('cnt').querySelectorAll('.tk').forEach((d,i)=>d.onclick=()=>show(e[i][0]));
  g('det').innerHTML='click a token';
}
function show(t){
  g('det').innerHTML=`<b>${esc(t)}</b> — ${where[t].length}x<br>`+
    where[t].map(([p,l])=>`pos ${p} <span style="font:12px monospace">${esc(POS[p])}</span> · layer ${l}`)
    .join('<br>');
}
setup(); draw();
</script>
""".replace("__D__",json.dumps(D)).replace("__L__",json.dumps(L)) \
   .replace("__POS__",json.dumps(POS)).replace("__K__",str(K))
display(HTML(html))

Findings:

* Assimetry prior to disambiguation consistent with stereotypes

  Example id: 0 (ambiguous "secretary", A = "The man", C = "The woman")

  "C" (6)                 vs                  "A" (1)



* Stickyness of stereotype congruent tokens even after disambiguation

  Example id: 25 (disambiguous "bad at math", B = "The boy", C = "The girl")

  "B" (5)                 vs                  "C" (3)



Layers 14-31 / Top-K = 2 / Final two positions / Deleted non-gendered tokens
